In [0]:
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# ==========================================
# BRONZE LAYER - IoT Telemetry
# DEV mode: GitHub Raw (free, for portfolio)
# PROD mode: S3 + Auto Loader (uncomment when infra is deployed)
# ==========================================

# --- Option A: DEV / Portfolio (current) ---
github_raw_url = "https://raw.githubusercontent.com/lynxiondev/iot-telemetry-data-platform/main/data/raw/raw_telemetry_dirty.csv"

print("⏳ Reading CSV from GitHub Raw (pandas bridge for Serverless)...")
df_pandas = pd.read_csv(github_raw_url)
print(f"✅ In-memory read: {len(df_pandas)} rows")

df_bronze = spark.createDataFrame(df_pandas)

# --- Option B: PROD (AWS) - ready for Terraform deployment ---
# This is what I'll use in prod. Requires s3://lynxion-iot-raw-prod/ + external location
# df_bronze = spark.readStream.format("cloudFiles") \
#     .option("cloudFiles.format", "json") \
#     .option("cloudFiles.schemaLocation", "/Volumes/main/checkpoints/bronze_schema") \
#     .load("s3://lynxion-iot-raw-prod/telemetry/")

# --- Audit columns - senior pattern ---
table_name = "bronze_telemetry"

df_bronze_final = df_bronze \
    .withColumn("ingest_timestamp", current_timestamp()) \
    .withColumn("_source", lit(github_raw_url)) # In prod: input_file_name()

df_bronze_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✅ SUCCESS! Table saved: {table_name}")
display(spark.table(table_name).limit(5))

In [0]:
# Consultar el historial de versiones de nuestra tabla
display(spark.sql("DESCRIBE HISTORY bronze_telemetry"))

In [0]:
# Viajar en el tiempo - ver cómo estaba antes del último overwrite
# display(spark.sql("SELECT * FROM bronze_telemetry VERSION AS OF 0 LIMIT 5"))